# Hybrid NN-GARCH Training Pipeline

This notebook uses:
- `model.py`
- `engine.py`
- `data_utils.py`
- `visualization.py`


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from model import HybridGarch
from data_utils import GARCHGenerator, MultiGARCHDataset, split_series_indices, standardize_returns, scale_variances
from engine import train_hybrid_garch, eval_log_mse, arch_baseline_log_mse_per_series
from visualization import plot_training_curves, plot_predictions_analysis


In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

N_SERIES = 2000
N_SAMPLES = 2000
WINDOW_SIZE = 90
TRAIN_SPLIT = 0.6
VAL_SPLIT = 0.2

HIDDEN_DIM = 32
BATCH_SIZE = 64
N_EPOCHS = 10
LR = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
true_omegas = np.random.uniform(0.001, 0.05, size=N_SERIES)
true_betas = np.random.uniform(0.01, 0.85, size=N_SERIES)
true_sums = np.random.uniform(true_betas + 0.01, 0.99)
true_alphas = true_sums - true_betas

all_returns = []
all_variances = []
for j in range(N_SERIES):
    gen = GARCHGenerator(true_omegas[j], true_alphas[j], true_betas[j], n_samples=N_SAMPLES, seed=SEED + j)
    r, s2 = gen.generate_series()
    all_returns.append(r)
    all_variances.append(s2)

all_returns = torch.stack(all_returns, dim=0)
all_variances = torch.stack(all_variances, dim=0)

tr_ids, va_ids, te_ids = split_series_indices(N_SERIES, train_split=TRAIN_SPLIT, val_split=VAL_SPLIT, seed=SEED)

train_returns_raw = all_returns[tr_ids]
val_returns_raw = all_returns[va_ids]
test_returns_raw = all_returns[te_ids]

train_vars_raw = all_variances[tr_ids]
val_vars_raw = all_variances[va_ids]
test_vars_raw = all_variances[te_ids]

mean_r = train_returns_raw.mean()
std_r = train_returns_raw.std()

train_returns = standardize_returns(train_returns_raw, mean_r, std_r)
val_returns = standardize_returns(val_returns_raw, mean_r, std_r)
test_returns = standardize_returns(test_returns_raw, mean_r, std_r)

train_vars = scale_variances(train_vars_raw, std_r)
val_vars = scale_variances(val_vars_raw, std_r)
test_vars = scale_variances(test_vars_raw, std_r)


In [ ]:
train_ds = MultiGARCHDataset(train_returns, train_vars, window_size=WINDOW_SIZE, stride=10)
val_ds = MultiGARCHDataset(val_returns, val_vars, window_size=WINDOW_SIZE, stride=10)
test_ds = MultiGARCHDataset(test_returns, test_vars, window_size=WINDOW_SIZE, stride=10)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Samples -> train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')


In [ ]:
model = HybridGarch(hidden=HIDDEN_DIM).to(device)
history = train_hybrid_garch(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS,
    lr=LR,
    device=device,
    ckpt_path='hybrid_garch_pretrained_synth.pt',
)


In [ ]:
val_log_mse = eval_log_mse(model, val_loader, device)
test_log_mse = eval_log_mse(model, test_loader, device)
print(f'Validation log-MSE: {val_log_mse:.6f}')
print(f'Test log-MSE: {test_log_mse:.6f}')

try:
    arch_val = arch_baseline_log_mse_per_series(val_returns[:100], val_vars[:100], WINDOW_SIZE, fit_ratio=0.6)
    print(f'ARCH baseline val log-MSE: {arch_val:.6f}')
except Exception as e:
    print('ARCH baseline skipped:', repr(e))


In [ ]:
plot_training_curves(history, test_loss=test_log_mse, title='Hybrid NN-GARCH training')
_ = plot_predictions_analysis(model, val_loader, device, n_examples=500, save_path='nn_garch_diagnostics.png')
